In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import PCA
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
kmeans = KMeans(featuresCol="features_scaled_final", k=6, seed=42)
best_model = kmeans.fit(df_final_kmeans)
predictions = best_model.transform(df_final_kmeans)
centers = best_model.clusterCenters()
df_centroids = pd.DataFrame(centers, columns=final_features)
df_centroids.index.name = "Cluster_ID"
print("\n--- BẢNG TỌA ĐỘ TÂM CỤM (CENTROIDS) ---")
print(df_centroids.round(3).to_string())
pca_viz = PCA(k=6, inputCol="features_scaled_final", outputCol="pca_2d")
predictions_2d = pca_viz.fit(predictions).transform(predictions)
print("Đang xử lý dữ liệu đồ họa...")
df_viz = predictions_2d.select("pca_2d", "prediction").sample(fraction=0.05, seed=42).toPandas()
df_viz['PC1'] = df_viz['pca_2d'].apply(lambda x: x[0])
df_viz['PC2'] = df_viz['pca_2d'].apply(lambda x: x[1])
plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=df_viz,
    x='PC1',
    y='PC2',
    hue='prediction',
    palette='Set1',
    alpha=0.7,
    s=30
)
plt.title(" Biểu đồ phân tán (K=6)", fontsize=14, fontweight='bold')
plt.xlabel("Trục phân tách chính (PC1)", fontsize=12)
plt.ylabel("Trục phân tách phụ (PC2)", fontsize=12)
plt.legend(title="Trạng thái (Cluster)", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("Biểu đồ phân tán.png", dpi=300, bbox_inches='tight', pad_inches=0.2)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
cluster_counts = predictions.groupBy("prediction").count().toPandas()
total_points = cluster_counts['count'].sum()
cluster_counts['Tỷ lệ (%)'] = (cluster_counts['count'] / total_points) * 100
cluster_counts = cluster_counts.sort_values(by="prediction").reset_index(drop=True)
cluster_counts.rename(columns={"prediction": "Cluster_ID", "count": "Số lượng (điểm dữ liệu)"}, inplace=True)
print("\n--- BẢNG PHÂN BỔ THỜI GIAN VẬN HÀNH ---")
print(cluster_counts.round(2).to_string(index=False))
plt.figure(figsize=(9, 9))
explode = [0.05] * len(cluster_counts)
plt.pie(
    cluster_counts['Tỷ lệ (%)'],
    labels=[f"Cluster {i}" for i in cluster_counts['Cluster_ID']],
    autopct='%1.2f%%',
    startangle=140,
    colors=sns.color_palette("Set1", n_colors=len(cluster_counts)),
    explode=explode,
    shadow=True
)
plt.title("Tỷ trọng thời gian phân bổ các trạng thái máy nén khí (K=6)", fontsize=15, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig("Tỷ lệ cluster.png", dpi=300, bbox_inches='tight', pad_inches=0.2)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import pi
categories = final_features
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]
plt.figure(figsize=(12, 10))
ax = plt.subplot(111, polar=True)
ax.set_theta_offset(pi / 2)
ax.set_theta_direction(-1)
plt.xticks(angles[:-1], categories, size=11, fontweight='bold')
ax.set_rlabel_position(0)
plt.yticks([-2, -1, 0, 1, 2, 4], ["-2", "-1", "0", "1", "2", "4"], color="grey", size=9)
plt.ylim(-3, 6)
colors = sns.color_palette("Set1", n_colors=6)
for i in range(len(df_centroids)):
    values = df_centroids.iloc[i].values.flatten().tolist()
    values += values[:1]
    ax.plot(angles, values, color=colors[i], linewidth=2.5, linestyle='solid', label=f"Cluster {i}")
    ax.fill(angles, values, color=colors[i], alpha=0.15)
plt.title("Hồ sơ đặc trưng (Radar Chart) của 6 trạng thái máy nén khí", size=16, fontweight='bold', y=1.08)
plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), title="Trạng thái (Cluster)", fontsize=10)
plt.grid(color='#AAAAAA', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.savefig("radar_chart_clusters.png", dpi=300, bbox_inches='tight', pad_inches=0.2)
plt.show()